## 神经网络 neural networks

### 定义网络

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 看一下 默认的 Net 是什么

class Net(nn.Module): 
    def __init__(self):
        super(Net, self).__init__()
        self.conv2d1 = nn.Conv2d(1, 6, 5)
        self.conv2d2 = nn.Conv2d(6, 16, 5)
        # 仿射函数 y = Wx + b
        self.f1 = nn.Linear(400, 120)
        self.f2 = nn.Linear(120, 84)
        self.f3 = nn.Linear(84, 10)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv2d1(x)), (2, 2))
        x = F.max_pool2d(F.relu(self.conv2d2(x)), (2, 2))
        x = x.view(-1, self.num_flat_features(x))
        x = F.relu(self.f1(x))
        x = F.relu(self.f2(x))
        x = self.f3(x)
        return x
    def num_flat_features(self, x):
        size = x.size()[1:]  # all dimensions except the batch dimension
        num_features = 1
        for s in size:
            num_features *= s
        return num_features
    
net = Net()
print(net)



Net(
  (conv2d1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2d2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (f1): Linear(in_features=400, out_features=120, bias=True)
  (f2): Linear(in_features=120, out_features=84, bias=True)
  (f3): Linear(in_features=84, out_features=10, bias=True)
)


### 笔记：`self` / `super` / 动态图（对初学者很重要）

**`self`**
- `self` 就是调用该方法的实例本身。`net.forward(x)` 底层等价于 `Net.forward(net, x)`——Python 会把调用者自动作为第一个参数传入，方法定义时必须显式写出这个形参来接收它。
- `__init__` 里 `self.conv1 = nn.Conv2d(...)` 把层绑定到"这个实例"上；`forward` 里 `self.conv1(input)` 从同一个实例取回来用。没有 `self`，两个方法之间就没法共享这些层。

**`super().__init__()`**
- `super(Net, self).__init__()` 是显式写法；`super().__init__()` 是 Python 3 的等价简写（编译器自动用方法所在类的隐藏 `__class__` 和第一个参数补全）。两者效果相同，但无参 `super()` 只能在类方法体内直接用。
- 这一步必须调用，因为 `nn.Module.__init__` 会初始化 `_parameters`/`_modules` 等内部字典，跳过它会导致后面 `self.conv1 = ...` 这样的子层注册失败。

**动态图（define-by-run）**
- `forward` 只是普通 Python 代码。每次调用 `net(input)`，PyTorch 都边执行边记录：`conv1(input)` → `ConvolutionBackward0`，`F.relu(...)` → `ReluBackward0`，`F.max_pool2d(...)` → `MaxPool2DWithIndicesBackward0`，`nn.Linear` 内部的矩阵乘 → `AddmmBackward0`……一路把这些节点连起来，直到各层的 `weight`/`bias`（`AccumulateGrad` 叶子节点）。
- 只要参与运算的张量里有一个 `requires_grad=True`（这里是各层权重，默认就是 `True`），输出就会带上 `grad_fn`，说明这张图已经建好了。
- 这张图是**一次性、跑一次建一次**的：`backward()` 沿图反向走一遍算出梯度后，图默认就被释放（除非 `retain_graph=True`）；下次再 `forward` 会重新构建一张新的图。这与静态图（先定义好整张图再反复喂数据）是本质区别。


In [12]:
params = list(net.parameters())
print('params = ', params)
print('params[0].size() = ', params[0].size())

params =  [Parameter containing:
tensor([[[[ 0.1196,  0.1116, -0.0576,  0.0775,  0.0011],
          [ 0.1196, -0.1566, -0.1121,  0.0533,  0.0580],
          [-0.1425,  0.0732, -0.0836, -0.0992, -0.0730],
          [ 0.1549,  0.0674,  0.0416, -0.1513, -0.1873],
          [-0.0072, -0.1039, -0.0896,  0.1171, -0.1313]]],


        [[[ 0.1815, -0.1373,  0.0625, -0.0285,  0.0195],
          [-0.0786,  0.0167,  0.1629,  0.1620, -0.0901],
          [ 0.0844,  0.0371, -0.0583, -0.0554,  0.1905],
          [-0.1719,  0.1958,  0.0112,  0.1500, -0.0210],
          [ 0.1393, -0.1699, -0.1661,  0.0705,  0.0170]]],


        [[[ 0.0265, -0.0722, -0.0235,  0.0944,  0.0015],
          [-0.0580, -0.0625,  0.1598,  0.1282,  0.0218],
          [-0.1339,  0.0936, -0.1151,  0.1032,  0.1811],
          [ 0.0716, -0.1328, -0.1683,  0.0650,  0.0314],
          [-0.1988,  0.0208, -0.0371,  0.1672,  0.1504]]],


        [[[ 0.1827, -0.1056, -0.1794, -0.0549, -0.1224],
          [ 0.1108, -0.1420,  0.1210,  0.13

In [5]:
# 完整最小示例：创建网络 -> 造输入 -> 使用网络 -> 看输出，四步缺一不可
input = torch.randn(1, 1, 32, 32)   # ① 造一个假输入：1张图，1通道，32x32
out = net(input)                     # ② 使用网络（前向传播，同时动态建图）
print(out)                           # ③ 看输出：10个类别的原始分数

# ④ 验证动态图确实生成了：out 带有 requires_grad 和 grad_fn
print('\nout.requires_grad:', out.requires_grad)
print('out.grad_fn      :', out.grad_fn)

# 反向传播前，所有参数的梯度都是 None
print('\n反向传播前 conv2d1.weight.grad =', net.conv2d1.weight.grad)

# 沿着刚建好的动态图反向传播一次
out.sum().backward()

# 反向传播后，梯度被填充进每个参数的 .grad
print('反向传播后 conv2d1.weight.grad.shape =', net.conv2d1.weight.grad.shape)


tensor([[ 0.1250,  0.0207,  0.0871,  0.0376,  0.0129, -0.0667, -0.1160, -0.0852,
         -0.1092,  0.0930]], grad_fn=<AddmmBackward0>)

out.requires_grad: True
out.grad_fn      : <AddmmBackward0 object at 0x12536ea40>

反向传播前 conv2d1.weight.grad = tensor([[[[-0.0414,  0.1515,  0.0489, -0.0509, -0.1401],
          [-0.1299, -0.0570, -0.0724,  0.0178,  0.0819],
          [-0.0463, -0.1414, -0.1177,  0.0114,  0.0364],
          [ 0.1029,  0.0866, -0.0708, -0.1214,  0.0135],
          [ 0.0240,  0.0558, -0.0830, -0.0913, -0.0499]]],


        [[[ 0.0597, -0.0197, -0.1523,  0.0948,  0.0531],
          [-0.0254, -0.1538,  0.0019,  0.0074,  0.0875],
          [-0.0553, -0.0152,  0.0790, -0.0576, -0.0224],
          [-0.0040, -0.0136,  0.1091,  0.0020, -0.1547],
          [ 0.1337, -0.0072,  0.0116,  0.1058, -0.0270]]],


        [[[ 0.1304,  0.1565, -0.1018, -0.0513,  0.0233],
          [ 0.0159,  0.1521, -0.0878,  0.0345,  0.1124],
          [ 0.0569,  0.0950,  0.0085,  0.0705,  0.0349],
   

In [19]:
# 使用新的输入 从新计算
input = torch.randn(1, 1, 32, 32)
out = net(input)
print(out)

tensor([[ 0.1429,  0.0527,  0.1004,  0.0635,  0.0209, -0.0851, -0.0788, -0.0637,
         -0.0592,  0.0791]], grad_fn=<AddmmBackward0>)


net.zero_grad()
out.backward(torch.randn(size = (1, 10)))
print(torch.randn(size = (1, 10)))

### 损失函数

In [25]:
output = net(input)
print(output.shape)
target = torch.randn(10)  # 随机值作为样例
target = target.view(1, -1)  # 使target和output的shape相同
criterion = nn.MSELoss()

loss = criterion(output, target)
print(loss)

torch.Size([1, 10])
tensor(1.1977, grad_fn=<MseLossBackward0>)


In [26]:
print(loss.grad_fn)  # MSELoss
print(loss.grad_fn.next_functions[0][0])  # Linear
print(loss.grad_fn.next_functions[0][0].next_functions[0][0])  # ReLU

### 反向传播

In [28]:
net.zero_grad()     # 清除梯度

print('conv2d1.bias.grad before backward')
print(net.conv2d1.bias.grad)

loss.backward()

print('conv2d1.bias.grad after backward')
print(net.conv2d1.bias.grad)

conv2d1.bias.grad before backward
None
conv2d1.bias.grad after backward
tensor([ 0.0041, -0.0175, -0.0208,  0.0047,  0.0082,  0.0257])


### 更新权重

In [30]:
import torch.optim as optim

# create your optimizer
optimizer = optim.SGD(net.parameters(), lr=0.01)

# in your training loop:
optimizer.zero_grad()   # zero the gradient buffers
output = net(input)
loss = criterion(output, target)
loss.backward()
optimizer.step()    # Does the update